In [1]:
import os
from google.colab import userdata, drive

# Kaggle
os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_KEY')
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')

# WANDB
!pip install wandb -q

import wandb
wandb.login(key=userdata.get('WANDB_API_KEY'))

print("Setup completed successfully.")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: akeke23 (akeke23-free-university-of-tbilisi-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Setup completed successfully.


**data download**

In [2]:
!pip install -q --upgrade kaggle

!kaggle competitions download -c challenges-in-representation-learning-facial-expression-recognition-challenge
!unzip -q -o challenges-in-representation-learning-facial-expression-recognition-challenge.zip
!ls -la

print("Data ready!")

challenges-in-representation-learning-facial-expression-recognition-challenge.zip: Skipping, found more recently modified local copy (use --force to force download)
total 1194888
drwxr-xr-x 1 root root      4096 Jun 16 19:32 .
drwxr-xr-x 1 root root      4096 Jun 16 16:07 ..
-rw-r--r-- 1 root root  44804427 Jun 16 16:58 16_ResNet18_RandomErasing_best.pt
-rw-r--r-- 1 root root  45322277 Jun 16 17:28 17_ResNet18_RC_LS_best.pt
-rw-r--r-- 1 root root  45322537 Jun 16 18:51 18_ResNet18_RC_LB_W_best.pt
-rw-r--r-- 1 root root  45323447 Jun 16 18:14 18_ResNet18_RC_LS_Weighted_best.pt
-rw-r--r-- 1 root root  45322927 Jun 16 19:29 19_ResNet18_RC_LB_W_v2_best.pt
-rw-r--r-- 1 root root 299063632 Dec 11  2019 challenges-in-representation-learning-facial-expression-recognition-challenge.zip
drwxr-xr-x 4 root root      4096 Jun  4 13:39 .config
-rw-r--r-- 1 root root      7178 Dec 11  2019 example_submission.csv
-rw-r--r-- 1 root root  96433867 Dec 11  2019 fer2013.tar.gz
-rw-r--r-- 1 root root 30107

**active device**

In [3]:
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Active Device:", device)

Active Device: cuda


**data preprocessing**

In [4]:
EMOTIONS = ['Angry', 'Disgust', 'Fear', 'Happy', 'Sad', 'Surprise', 'Neutral']

def parse_pixels(pixel_series):
    raw_arrays = np.array([np.array(p.split(), dtype=np.uint8) for p in pixel_series])
    return raw_arrays.reshape(-1, 48, 48)

# read data and strip spaces
df = pd.read_csv('/content/icml_face_data.csv')
df.columns = df.columns.str.strip()
df['Usage'] = df['Usage'].str.strip()

# create splits
train_mask = df['Usage'] == 'Training'
val_mask   = df['Usage'] == 'PublicTest'
test_mask  = df['Usage'] == 'PrivateTest'

X_train, y_train = parse_pixels(df.loc[train_mask, 'pixels']), df.loc[train_mask, 'emotion'].values
X_val,   y_val   = parse_pixels(df.loc[val_mask, 'pixels']),   df.loc[val_mask, 'emotion'].values
X_test,  y_test  = parse_pixels(df.loc[test_mask, 'pixels']),  df.loc[test_mask, 'emotion'].values

print(f"Data Shapes - Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}")

Data Shapes - Train: (28709, 48, 48) | Val: (3589, 48, 48) | Test: (3589, 48, 48)


In [ ]:
from torchvision import transforms, models
from torch.utils.data import WeightedRandomSampler
from PIL import Image

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

class FERDataset(Dataset):
    def __init__(self, imgs, lbls, transform=None):
        self.imgs      = imgs
        self.lbls      = lbls
        self.transform = transform

    def __len__(self):
        return len(self.imgs)

    def __getitem__(self, idx):
        img = Image.fromarray(self.imgs[idx])
        if self.transform:
            img = self.transform(img)
        return img, int(self.lbls[idx])

# weighted sampler
class_counts   = pd.Series(y_train).value_counts().sort_index().values
sample_weights = 1.0 / class_counts[y_train]
sampler        = WeightedRandomSampler(
    torch.tensor(sample_weights, dtype=torch.float),
    num_samples=len(sample_weights),
    replacement=True
)

train_dataset = FERDataset(X_train, y_train, transform=train_transform)
val_dataset   = FERDataset(X_val,   y_val,   transform=val_transform)
test_dataset  = FERDataset(X_test,  y_test,  transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=64, sampler=sampler,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=64, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

print(f'Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}')

Train: 28709 | Val: 3589 | Test: 3589


In [ ]:
class FERResNet(nn.Module):
    def __init__(self, num_classes=7, dropout_rate=0.5, freeze_backbone=False):
        super().__init__()

        self.backbone = models.resnet18(weights='IMAGENET1K_V1')

        if freeze_backbone:
            for param in self.backbone.parameters():
                param.requires_grad = False

        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Sequential(
            nn.Dropout(p=dropout_rate),
            nn.Linear(in_features, num_classes)
        )

    def forward(self, x):
        return self.backbone(x)


m     = FERResNet(freeze_backbone=False)
total = sum(p.numel() for p in m.parameters() if p.requires_grad)
print(f'ResNet18 (full) trainable params: {total:,}')

m_frozen = FERResNet(freeze_backbone=True)
total_frozen = sum(p.numel() for p in m_frozen.parameters() if p.requires_grad)
print(f'ResNet18 (frozen) trainable params: {total_frozen:,}')

ResNet18 (full) trainable params: 11,180,103
ResNet18 (frozen) trainable params: 3,591


In [ ]:
import math

test_model = FERResNet(freeze_backbone=False).to(device)
criterion  = nn.CrossEntropyLoss()

images, labels = next(iter(train_loader))
images, labels = images.to(device), labels.to(device)


print('FORWARD CHECK')
with torch.no_grad():
    out       = test_model(images)
    init_loss = criterion(out, labels)
print(f'Initial loss:   {init_loss.item():.4f}')
print(f'Expected (ln7): {math.log(7):.4f}')
print(f'Difference:     {abs(init_loss.item() - math.log(7)):.4f}  (< 0.1 = OK)')

print('\n')
print('BACKWARD CHECK ')
opt = torch.optim.Adam(test_model.parameters(), lr=1e-4)
for step in range(200):
    opt.zero_grad()
    out  = test_model(images)
    loss = criterion(out, labels)
    loss.backward()
    opt.step()
    if step % 50 == 0 or step == 199:
        acc       = (out.argmax(1) == labels).float().mean().item()
        grad_norm = sum(p.grad.data.norm(2).item() ** 2
                        for p in test_model.parameters() if p.grad is not None) ** 0.5
        print(f'Step {step:3d} | Loss: {loss.item():.4f} | Acc: {acc:.3f} | GradNorm: {grad_norm:.3f}')

FORWARD CHECK
Initial loss:   2.4414
Expected (ln7): 1.9459
Difference:     0.4955  (< 0.1 = OK)


BACKWARD CHECK 
Step   0 | Loss: 2.1774 | Acc: 0.141 | GradNorm: 12.316
Step  50 | Loss: 0.0039 | Acc: 1.000 | GradNorm: 0.057
Step 100 | Loss: 0.0020 | Acc: 1.000 | GradNorm: 0.034
Step 150 | Loss: 0.0013 | Acc: 1.000 | GradNorm: 0.022
Step 199 | Loss: 0.0011 | Acc: 1.000 | GradNorm: 0.016


**helper funcs**

In [ ]:
class EarlyStopping:
    def __init__(self, patience=7, min_delta=0.001):
        self.patience     = patience
        self.min_delta    = min_delta
        self.best_acc     = 0.0
        self.counter      = 0
        self.best_weights = None
        self.should_stop  = False

    def step(self, val_acc, model):
        if val_acc > self.best_acc + self.min_delta:
            self.best_acc     = val_acc
            self.counter      = 0
            self.best_weights = {k: v.cpu().clone()
                                 for k, v in model.state_dict().items()}
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.should_stop = True

    def restore_best(self, model):
        if self.best_weights:
            model.load_state_dict(self.best_weights)


def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct, total, total_grad_norm = 0.0, 0, 0, 0.0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs   = model(images)
        loss      = criterion(outputs, labels)
        loss.backward()
        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        total_grad_norm += grad_norm.item()
        optimizer.step()
        total_loss += loss.item() * images.size(0)
        correct    += (outputs.argmax(1) == labels).sum().item()
        total      += labels.size(0)
    return total_loss / total, correct / total, total_grad_norm / len(loader)


def evaluate_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.inference_mode():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs    = model(images)
            loss       = criterion(outputs, labels)
            total_loss += loss.item() * images.size(0)
            correct    += (outputs.argmax(1) == labels).sum().item()
            total      += labels.size(0)
    return total_loss / total, correct / total


def run_experiment(model, run_name, config, train_loader, val_loader,
                   device, use_early_stopping=False):
    wandb.init(
        project='fer-challenge',
        name=run_name,
        group=config['architecture'],
        config=config,
        reinit=True
    )

    model     = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=config['lr'],
        weight_decay=config.get('weight_decay', 0.0)
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=5
    )

    stopper      = EarlyStopping(patience=7, min_delta=0.001) if use_early_stopping else None
    best_val_acc = 0.0

    for epoch in range(1, config['epochs'] + 1):
        train_loss, train_acc, grad_norm = train_epoch(model, train_loader, criterion, optimizer, device)
        val_loss,   val_acc              = evaluate_epoch(model, val_loader, criterion, device)

        scheduler.step(val_acc)
        current_lr = optimizer.param_groups[0]['lr']
        gap        = train_acc - val_acc

        log_dict = {
            'epoch':      epoch,
            'train_loss': train_loss,
            'val_loss':   val_loss,
            'train_acc':  train_acc,
            'val_acc':    val_acc,
            'acc_gap':    gap,
            'grad_norm':  grad_norm,
            'lr':         current_lr,
        }
        if stopper is not None:
            log_dict['early_stop_counter'] = stopper.counter

        wandb.log(log_dict)

        print(f'Ep {epoch:02d}/{config["epochs"]} | '
              f'Train {train_acc*100:.1f}% ({train_loss:.4f}) | '
              f'Val {val_acc*100:.1f}% ({val_loss:.4f}) | '
              f'Gap {gap*100:.1f}% | GradNorm {grad_norm:.3f}')

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), f'/content/{run_name}_best.pt')

        if stopper is not None:
            stopper.step(val_acc, model)
            if stopper.should_stop:
                print(f'Early stopping at epoch {epoch} (best: {stopper.best_acc*100:.2f}%)')
                stopper.restore_best(model)
                break

    wandb.log({'epochs_trained': epoch, 'best_val_acc': best_val_acc})
    wandb.finish()
    print(f'\n-> Best val acc: {best_val_acc*100:.2f}%')
    return best_val_acc

**training**

In [ ]:
config_frozen = {
    'architecture':    'ResNet18',
    'variant':         'Frozen_backbone',
    'lr':              1e-4,
    'batch_size':      64,
    'optimizer':       'Adam',
    'epochs':          15,
    'dropout_rate':    0.5,
    'weight_decay':    0.0,
    'freeze_backbone': True,
    'early_stopping':  False,
}

model_frozen = FERResNet(dropout_rate=0.5, freeze_backbone=True)
acc_frozen   = run_experiment(
    model_frozen, '12_ResNet18_Frozen', config_frozen,
    train_loader, val_loader, device,
    use_early_stopping=False
)

wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


Ep 01/15 | Train 17.8% (2.0581) | Val 26.1% (1.8568) | Gap -8.3% | GradNorm 4.222
Ep 02/15 | Train 22.6% (1.9360) | Val 29.8% (1.7951) | Gap -7.2% | GradNorm 4.093
Ep 03/15 | Train 25.0% (1.8750) | Val 34.9% (1.7243) | Gap -9.9% | GradNorm 4.089
Ep 04/15 | Train 27.1% (1.8279) | Val 34.7% (1.7214) | Gap -7.6% | GradNorm 4.008
Ep 05/15 | Train 28.8% (1.8026) | Val 35.5% (1.7071) | Gap -6.8% | GradNorm 4.029
Ep 06/15 | Train 29.8% (1.7831) | Val 35.8% (1.6915) | Gap -6.1% | GradNorm 4.033
Ep 07/15 | Train 30.3% (1.7727) | Val 35.4% (1.6891) | Gap -5.1% | GradNorm 4.006
Ep 08/15 | Train 31.1% (1.7608) | Val 37.2% (1.6715) | Gap -6.1% | GradNorm 4.001
Ep 09/15 | Train 31.2% (1.7539) | Val 36.9% (1.6765) | Gap -5.7% | GradNorm 3.986
Ep 10/15 | Train 31.9% (1.7424) | Val 37.7% (1.6609) | Gap -5.8% | GradNorm 3.986
Ep 11/15 | Train 31.9% (1.7429) | Val 38.0% (1.6550) | Gap -6.1% | GradNorm 3.955
Ep 12/15 | Train 32.6% (1.7334) | Val 38.5% (1.6355) | Gap -5.9% | GradNorm 4.003
Ep 13/15 | Train

acc_gap,▃▅▁▄▆▇█▇▇▇▇▇▇▆▆
best_val_acc,▁
epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
epochs_trained,▁
grad_norm,█▅▅▃▃▃▂▂▂▂▁▂▁▁▁
lr,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_acc,▁▃▄▅▆▇▇▇▇██████
train_loss,█▅▄▃▃▂▂▂▂▁▁▁▁▁▁
val_acc,▁▃▆▆▆▆▆▇▇▇▇████
val_loss,█▆▄▄▃▃▃▂▂▂▂▁▁▁▁
acc_gap,-0.06342



-> Best val acc: 39.12%


In [ ]:
config_pure_unfreeze = {
    'architecture':    'ResNet18',
    'variant':         'Unfrozen_backbone_pure',
    'lr':              1e-4,
    'batch_size':      64,
    'optimizer':       'Adam',
    'epochs':          15,
    'dropout_rate':    0.5,
    'weight_decay':    0.0,
    'freeze_backbone': False,
    'early_stopping':  False,
}

model_pure = FERResNet(dropout_rate=0.5, freeze_backbone=False)

acc_pure = run_experiment(
    model_pure, '13_ResNet18_Pure_Unfrozen', config_pure_unfreeze,
    train_loader, val_loader, device,
    use_early_stopping=False
)

Ep 01/15 | Train 53.5% (1.2493) | Val 60.9% (1.0688) | Gap -7.4% | GradNorm 7.386
Ep 02/15 | Train 67.4% (0.8866) | Val 63.4% (1.0102) | Gap 4.1% | GradNorm 5.575
Ep 03/15 | Train 71.8% (0.7653) | Val 63.8% (0.9909) | Gap 8.0% | GradNorm 5.226
Ep 04/15 | Train 75.7% (0.6694) | Val 63.8% (1.0220) | Gap 11.8% | GradNorm 5.132
Ep 05/15 | Train 78.7% (0.5920) | Val 64.7% (1.0326) | Gap 13.9% | GradNorm 5.141
Ep 06/15 | Train 81.5% (0.5206) | Val 66.0% (1.0838) | Gap 15.5% | GradNorm 5.272
Ep 07/15 | Train 83.4% (0.4656) | Val 66.3% (1.0935) | Gap 17.1% | GradNorm 5.433
Ep 08/15 | Train 85.6% (0.4164) | Val 66.4% (1.1140) | Gap 19.2% | GradNorm 5.423
Ep 09/15 | Train 87.4% (0.3598) | Val 67.3% (1.1340) | Gap 20.1% | GradNorm 5.519
Ep 10/15 | Train 88.5% (0.3300) | Val 67.8% (1.1758) | Gap 20.7% | GradNorm 5.586
Ep 11/15 | Train 90.3% (0.2862) | Val 67.1% (1.2352) | Gap 23.1% | GradNorm 5.529
Ep 12/15 | Train 91.0% (0.2649) | Val 67.4% (1.2521) | Gap 23.6% | GradNorm 5.549
Ep 13/15 | Train 9

acc_gap,▁▃▄▅▅▆▆▇▇▇▇▇███
best_val_acc,▁
epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
epochs_trained,▁
grad_norm,█▂▁▁▁▁▂▂▂▂▂▂▂▂▂
lr,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_acc,▁▃▄▅▅▆▆▇▇▇▇████
train_loss,█▆▅▄▄▃▃▂▂▂▂▁▁▁▁
val_acc,▁▃▄▄▅▆▆▆▇█▇▇▇█▇
val_loss,▂▁▁▂▂▃▃▃▄▄▅▆▇▇█
acc_gap,0.26208



-> Best val acc: 67.96%


In [ ]:
config_full_es = {
    'architecture':    'ResNet18',
    'variant':         'Full_finetune_ES_WD',
    'lr':              1e-4,
    'batch_size':      64,
    'optimizer':       'Adam',
    'epochs':          25,
    'dropout_rate':    0.5,
    'weight_decay':    1e-4,
    'freeze_backbone': False,
    'early_stopping':  True,
}

model_full_es = FERResNet(dropout_rate=0.5, freeze_backbone=False)
acc_full_es   = run_experiment(
    model_full_es, '14_ResNet18_FullFinetune_ES_WD', config_full_es,
    train_loader, val_loader, device,
    use_early_stopping=True
)

Ep 01/25 | Train 52.9% (1.2510) | Val 60.6% (1.0632) | Gap -7.7% | GradNorm 8.478
Ep 02/25 | Train 67.5% (0.8828) | Val 62.7% (1.0088) | Gap 4.8% | GradNorm 5.986
Ep 03/25 | Train 71.8% (0.7633) | Val 63.8% (0.9834) | Gap 7.9% | GradNorm 5.440
Ep 04/25 | Train 75.8% (0.6659) | Val 63.6% (1.0629) | Gap 12.2% | GradNorm 5.191
Ep 05/25 | Train 78.4% (0.5920) | Val 65.3% (1.0235) | Gap 13.2% | GradNorm 5.243
Ep 06/25 | Train 81.0% (0.5267) | Val 64.8% (1.0834) | Gap 16.2% | GradNorm 5.365
Ep 07/25 | Train 83.5% (0.4719) | Val 66.6% (1.0203) | Gap 16.9% | GradNorm 5.470
Ep 08/25 | Train 85.1% (0.4223) | Val 65.4% (1.0817) | Gap 19.7% | GradNorm 5.610
Ep 09/25 | Train 87.0% (0.3735) | Val 67.1% (1.1058) | Gap 19.9% | GradNorm 5.726
Ep 10/25 | Train 88.7% (0.3302) | Val 66.5% (1.1503) | Gap 22.3% | GradNorm 5.752
Ep 11/25 | Train 89.4% (0.3063) | Val 67.0% (1.1638) | Gap 22.4% | GradNorm 5.870
Ep 12/25 | Train 90.6% (0.2728) | Val 66.6% (1.1798) | Gap 24.0% | GradNorm 5.812
Ep 13/25 | Train 9

acc_gap,▁▃▄▅▅▅▆▆▆▇▇▇▇▇▇▇█████████
best_val_acc,▁
early_stop_counter,▁▁▁▁▂▁▂▁▂▁▂▃▅▆▇█▁▂▃▅▆▇█▁▂
epoch,▁▁▂▂▂▂▃▃▃▄▄▄▅▅▅▅▆▆▆▇▇▇▇██
epochs_trained,▁
grad_norm,█▅▄▄▄▄▄▄▅▅▅▅▅▅▅▄▃▃▃▃▃▃▂▁▁
lr,██████████████▃▃▃▃▃▃▃▁▁▁▁
train_acc,▁▃▄▄▅▅▆▆▆▆▇▇▇▇▇▇█████████
train_loss,█▆▅▅▄▄▃▃▃▃▃▂▂▂▂▂▁▁▁▁▁▁▁▁▁
val_acc,▁▃▄▃▅▄▆▅▆▆▆▆▆▆▆█▇▇▇██▇███
+1,...



-> Best val acc: 69.21%


new loader

In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=64,
                         shuffle=True, num_workers=2, pin_memory=True)

weighted_loader = DataLoader(train_dataset, batch_size=64,
                            sampler=sampler, num_workers=2, pin_memory=True)

In [ ]:
config_no_sampler = {
    'architecture':    'ResNet18',
    'variant':         'Full_finetune_NoSampler_WD',
    'lr':              1e-4,
    'batch_size':      64,
    'optimizer':       'Adam',
    'epochs':          25,
    'dropout_rate':    0.5,
    'weight_decay':    1e-4,
    'freeze_backbone': False,
    'early_stopping':  True,
}

model_no_sampler = FERResNet(dropout_rate=0.5, freeze_backbone=False)
model_no_sampler = model_no_sampler.to(device)

acc_no_sampler = run_experiment(
    model_no_sampler, '15_ResNet18_FullFinetune_NoSampler_WD', config_no_sampler,
    train_loader, val_loader, device,
    use_early_stopping=True
)

wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


Ep 01/25 | Train 52.7% (1.2562) | Val 59.6% (1.0759) | Gap -6.9% | GradNorm 8.560
Ep 02/25 | Train 62.9% (0.9916) | Val 64.2% (0.9841) | Gap -1.3% | GradNorm 5.259
Ep 03/25 | Train 67.3% (0.8825) | Val 66.3% (0.9331) | Gap 1.0% | GradNorm 4.787
Ep 04/25 | Train 70.9% (0.7893) | Val 66.4% (0.9359) | Gap 4.5% | GradNorm 4.766
Ep 05/25 | Train 73.8% (0.7076) | Val 66.3% (0.9679) | Gap 7.5% | GradNorm 4.965
Ep 06/25 | Train 77.1% (0.6270) | Val 67.4% (0.9512) | Gap 9.8% | GradNorm 5.306
Ep 07/25 | Train 80.0% (0.5556) | Val 68.7% (0.9837) | Gap 11.3% | GradNorm 5.541
Ep 08/25 | Train 82.9% (0.4837) | Val 67.3% (1.0180) | Gap 15.6% | GradNorm 5.848
Ep 09/25 | Train 84.7% (0.4216) | Val 68.1% (1.0740) | Gap 16.6% | GradNorm 6.077
Ep 10/25 | Train 86.9% (0.3708) | Val 67.2% (1.1403) | Gap 19.7% | GradNorm 6.198
Ep 11/25 | Train 88.6% (0.3255) | Val 66.8% (1.1550) | Gap 21.8% | GradNorm 6.238
Ep 12/25 | Train 89.9% (0.2897) | Val 67.0% (1.1752) | Gap 22.9% | GradNorm 6.333
Ep 13/25 | Train 91.

acc_gap,▁▂▃▃▄▅▅▆▆▇▇▇▇█
best_val_acc,▁
early_stop_counter,▁▁▁▁▂▃▁▁▂▃▅▆▇█
epoch,▁▂▂▃▃▄▄▅▅▆▆▇▇█
epochs_trained,▁
grad_norm,█▂▁▁▁▂▂▃▃▄▄▄▄▂
lr,████████████▁▁
train_acc,▁▃▃▄▅▅▆▆▆▇▇▇▇█
train_loss,█▆▆▅▄▄▃▃▃▂▂▂▂▁
val_acc,▁▅▆▆▆▇█▇█▇▇▇▇█
+1,...



-> Best val acc: 68.71%


In [ ]:
from torchvision import transforms

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),

    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),

    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
    transforms.RandomErasing(p=0.3, scale=(0.02, 0.1))
])

In [ ]:
from torch.utils.data import DataLoader

train_dataset = FERDataset(X_train, y_train, transform=train_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

In [ ]:
config = {
    'architecture':    'ResNet18',
    'variant':         'ResNet18_RandomErasing',
    'lr':              1e-4,
    'batch_size':      64,
    'optimizer':       'Adam',
    'epochs':          25,
    'dropout_rate':    0.5,
    'weight_decay':    1e-4,
    'freeze_backbone': False,
    'early_stopping':  True,
}

model = FERResNet(dropout_rate=0.5, freeze_backbone=False)
model = model.to(device)

best_acc = run_experiment(
    model,
    '16_ResNet18_RandomErasing',
    config,
    train_loader,
    val_loader,
    device,
    use_early_stopping=True
)

wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


Ep 01/25 | Train 51.3% (1.2863) | Val 61.3% (1.0516) | Gap -10.0% | GradNorm 6.172
Ep 02/25 | Train 62.3% (1.0119) | Val 64.1% (0.9814) | Gap -1.8% | GradNorm 4.545
Ep 03/25 | Train 66.3% (0.9086) | Val 65.6% (0.9535) | Gap 0.7% | GradNorm 4.526
Ep 04/25 | Train 69.0% (0.8278) | Val 66.7% (0.9163) | Gap 2.3% | GradNorm 4.642
Ep 05/25 | Train 72.1% (0.7547) | Val 67.0% (0.9521) | Gap 5.2% | GradNorm 4.826
Ep 06/25 | Train 75.0% (0.6822) | Val 67.3% (0.9684) | Gap 7.7% | GradNorm 5.190
Ep 07/25 | Train 77.7% (0.6113) | Val 68.4% (0.9634) | Gap 9.3% | GradNorm 5.404
Ep 08/25 | Train 79.6% (0.5572) | Val 67.7% (1.0049) | Gap 11.9% | GradNorm 5.770
Ep 09/25 | Train 82.1% (0.4951) | Val 68.4% (1.0455) | Gap 13.7% | GradNorm 5.898
Ep 10/25 | Train 84.2% (0.4468) | Val 67.0% (1.0776) | Gap 17.2% | GradNorm 6.143
Ep 11/25 | Train 86.0% (0.3974) | Val 67.7% (1.1726) | Gap 18.3% | GradNorm 6.349
Ep 12/25 | Train 87.3% (0.3570) | Val 67.9% (1.1350) | Gap 19.4% | GradNorm 6.441
Ep 13/25 | Train 88.

acc_gap,▁▃▃▃▄▄▅▅▅▆▆▆▇▇▇██████████
best_val_acc,▁
early_stop_counter,▁▁▁▁▁▁▁▁▂▃▅▆▇█▁▂▃▅▁▂▃▅▆▇▁
epoch,▁▁▂▂▂▂▃▃▃▄▄▄▅▅▅▅▆▆▆▇▇▇▇██
epochs_trained,▁
grad_norm,▇▁▁▁▂▃▄▅▆▇███▅▅▅▅▅▅▅▄▅▅▅▅
lr,████████████▁▁▁▁▁▁▁▁▁▁▁▁▁
train_acc,▁▃▃▄▄▅▅▅▆▆▆▇▇▇███████████
train_loss,█▆▆▅▅▄▄▄▃▃▃▃▂▂▁▁▁▁▁▁▁▁▁▁▁
val_acc,▁▃▅▅▆▆▇▆▇▆▆▇▆██▇▇█▇▇▇▇██▇
+1,...



-> Best val acc: 69.69%


In [ ]:
class FERResNetV2(nn.Module):
    def __init__(self, num_classes=7, dropout_rate=0.5, freeze_backbone=False):
        super().__init__()

        self.backbone = models.resnet18(weights='IMAGENET1K_V1')

        if freeze_backbone:
            for param in self.backbone.parameters():
                param.requires_grad = False

        # better classifier: 512 -> 256 -> ReLU -> Dropout -> 7
        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Sequential(
            nn.Linear(in_features, 256),
            nn.ReLU(),
            nn.Dropout(p=dropout_rate),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        return self.backbone(x)

In [ ]:
import torch
import torch.nn as nn
import wandb

def run_experiment_v2(model, run_name, config, train_loader, val_loader, device, use_early_stopping=False):
    wandb.init(
        project='fer-challenge',
        name=run_name,
        group=config['architecture'],
        config=config,
        reinit=True
    )

    model = model.to(device)

    criterion = nn.CrossEntropyLoss(label_smoothing=config.get('label_smoothing', 0.0))

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=config['lr'],
        weight_decay=config.get('weight_decay', 0.0)
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=5
    )

    stopper      = EarlyStopping(patience=7, min_delta=0.001) if use_early_stopping else None
    best_val_acc = 0.0

    for epoch in range(1, config['epochs'] + 1):
        train_loss, train_acc, grad_norm = train_epoch(model, train_loader, criterion, optimizer, device)
        val_loss,   val_acc              = evaluate_epoch(model, val_loader, criterion, device)

        scheduler.step(val_acc)
        current_lr = optimizer.param_groups[0]['lr']
        gap        = train_acc - val_acc

        log_dict = {
            'epoch':      epoch,
            'train_loss': train_loss,
            'val_loss':   val_loss,
            'train_acc':  train_acc,
            'val_acc':    val_acc,
            'acc_gap':    gap,
            'grad_norm':  grad_norm,
            'lr':         current_lr,
        }
        if stopper is not None:
            log_dict['early_stop_counter'] = stopper.counter

        wandb.log(log_dict)

        print(f'Ep {epoch:02d}/{config["epochs"]} | '
              f'Train {train_acc*100:.1f}% ({train_loss:.4f}) | '
              f'Val {val_acc*100:.1f}% ({val_loss:.4f}) | '
              f'Gap {gap*100:.1f}% | GradNorm {grad_norm:.3f}')

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), f'/content/{run_name}_best.pt')

        if stopper is not None:
            stopper.step(val_acc, model)
            if stopper.should_stop:
                print(f'Early stopping at epoch {epoch} (best: {stopper.best_acc*100:.2f}%)')
                stopper.restore_best(model)
                break

    wandb.log({'epochs_trained': epoch, 'best_val_acc': best_val_acc})
    wandb.finish()
    print(f'\n-> Best val acc: {best_val_acc*100:.2f}%')
    return best_val_acc

In [ ]:
config_rc_ls = {
    'architecture':    'ResNet18',
    'variant':         'RicherClassifier_LabelSmoothing',
    'lr':              1e-4,
    'batch_size':      64,
    'optimizer':       'Adam',
    'epochs':          25,
    'dropout_rate':    0.5,
    'weight_decay':    1e-4,
    'freeze_backbone': False,
    'early_stopping':  True,
    'label_smoothing': 0.1,
}

model_rc_ls = FERResNetV2(dropout_rate=0.5, freeze_backbone=False)
acc_rc_ls   = run_experiment_v2(
    model_rc_ls, '17_ResNet18_RC_LS',
    config_rc_ls,
    train_loader, val_loader, device,
    use_early_stopping=True
)

Ep 01/25 | Train 53.4% (1.3948) | Val 60.2% (1.2520) | Gap -6.9% | GradNorm 3.301
Ep 02/25 | Train 63.1% (1.2123) | Val 64.3% (1.1826) | Gap -1.2% | GradNorm 2.968
Ep 03/25 | Train 67.0% (1.1389) | Val 66.2% (1.1656) | Gap 0.8% | GradNorm 2.949
Ep 04/25 | Train 70.2% (1.0834) | Val 67.3% (1.1372) | Gap 3.0% | GradNorm 3.100
Ep 05/25 | Train 72.8% (1.0300) | Val 67.1% (1.1390) | Gap 5.7% | GradNorm 3.288
Ep 06/25 | Train 75.5% (0.9811) | Val 67.7% (1.1393) | Gap 7.8% | GradNorm 3.557
Ep 07/25 | Train 78.1% (0.9305) | Val 68.7% (1.1348) | Gap 9.4% | GradNorm 3.795
Ep 08/25 | Train 80.2% (0.8908) | Val 68.0% (1.1715) | Gap 12.2% | GradNorm 4.021
Ep 09/25 | Train 82.4% (0.8481) | Val 69.3% (1.1590) | Gap 13.1% | GradNorm 4.210
Ep 10/25 | Train 84.4% (0.8064) | Val 68.3% (1.1870) | Gap 16.1% | GradNorm 4.315
Ep 11/25 | Train 85.8% (0.7766) | Val 67.8% (1.2057) | Gap 18.0% | GradNorm 4.482
Ep 12/25 | Train 87.8% (0.7412) | Val 68.0% (1.1973) | Gap 19.8% | GradNorm 4.463
Ep 13/25 | Train 88.7

acc_gap,▁▂▃▃▄▄▅▅▅▆▆▇▇▇▇█
best_val_acc,▁
early_stop_counter,▁▁▁▁▁▂▁▁▂▁▂▃▅▆▇█
epoch,▁▁▂▂▃▃▄▄▅▅▆▆▇▇██
epochs_trained,▁
grad_norm,▂▁▁▂▂▄▅▆▆▇▇▇███▅
lr,██████████████▁▁
train_acc,▁▃▃▄▄▅▅▆▆▆▇▇▇▇▇█
train_loss,█▆▆▅▅▄▄▃▃▃▂▂▂▂▂▁
val_acc,▁▄▆▆▆▇█▇█▇▇▇▇███
+1,...



-> Best val acc: 69.27%


In [ ]:
import torch
import torch.nn as nn
import wandb

def run_experiment_v3(model, run_name, config, train_loader, val_loader, device):
    wandb.init(
        project='fer-challenge',
        name=run_name,
        group=config['architecture'],
        config=config,
        reinit=True
    )

    model = model.to(device)

    criterion = nn.CrossEntropyLoss(label_smoothing=config.get('label_smoothing', 0.0))

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=config['lr'],
        weight_decay=config.get('weight_decay', 0.0)
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=2
    )

    use_early_stopping = config.get('early_stopping', False)
    patience = config.get('early_stop_patience', 4)
    stopper = EarlyStopping(patience=patience, min_delta=0.001) if use_early_stopping else None

    best_val_acc = 0.0

    for epoch in range(1, config['epochs'] + 1):
        train_loss, train_acc, grad_norm = train_epoch(model, train_loader, criterion, optimizer, device)
        val_loss,   val_acc              = evaluate_epoch(model, val_loader, criterion, device)

        scheduler.step(val_acc)
        current_lr = optimizer.param_groups[0]['lr']
        gap        = train_acc - val_acc

        log_dict = {
            'epoch':      epoch,
            'train_loss': train_loss,
            'val_loss':   val_loss,
            'train_acc':  train_acc,
            'val_acc':    val_acc,
            'acc_gap':    gap,
            'grad_norm':  grad_norm,
            'lr':         current_lr,
        }
        if stopper is not None:
            log_dict['early_stop_counter'] = stopper.counter

        wandb.log(log_dict)

        print(f'Ep {epoch:02d}/{config["epochs"]} | '
              f'Train {train_acc*100:.1f}% ({train_loss:.4f}) | '
              f'Val {val_acc*100:.1f}% ({val_loss:.4f}) | '
              f'Gap {gap*100:.1f}% | GradNorm {grad_norm:.3f}')

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), f'/content/{run_name}_best.pt')

        if stopper is not None:
            stopper.step(val_acc, model)
            if stopper.should_stop:
                print(f'Early stopping at epoch {epoch} (best: {stopper.best_acc*100:.2f}%)')
                stopper.restore_best(model)
                break

    wandb.log({'epochs_trained': epoch, 'best_val_acc': best_val_acc})
    wandb.finish()
    print(f'\n-> Best val acc: {best_val_acc*100:.2f}%')
    return best_val_acc

In [16]:
import torch
import pandas as pd
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import transforms

train_transform_final = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.3, scale=(0.02, 0.1))
])

train_dataset_final = FERDataset(X_train, y_train, transform=train_transform_final)

class_counts   = pd.Series(y_train).value_counts().sort_index().values
sample_weights = 1.0 / class_counts[y_train]
sampler_final  = WeightedRandomSampler(
    torch.tensor(sample_weights, dtype=torch.float),
    num_samples=len(sample_weights),
    replacement=True
)

train_loader_final = DataLoader(train_dataset_final, batch_size=64,
                                sampler=sampler_final, num_workers=2, pin_memory=True)
val_loader_final   = DataLoader(val_dataset,   batch_size=64, shuffle=False, num_workers=2, pin_memory=True)
test_loader_final  = DataLoader(test_dataset,  batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

print('Checked loaders ready!')

Checked loaders ready!


In [14]:
def create_experiment_loaders(X_train_data, y_train_labels, X_val_data, y_val_labels, X_test_data, y_test_labels,
                              train_transform_func, val_transform_func, config_dict):

    # Create datasets with specified transforms
    train_dataset = FERDataset(X_train_data, y_train_labels, transform=train_transform_func)
    val_dataset   = FERDataset(X_val_data,   y_val_labels,   transform=val_transform_func)
    test_dataset  = FERDataset(X_test_data,  y_test_labels,  transform=val_transform_func)

    batch_size = config_dict.get('batch_size', 64)

    # Create train_loader with or without weighted sampling
    if config_dict.get('weighted_sampling', False):
        class_counts = pd.Series(y_train_labels).value_counts().sort_index().values
        sample_weights = 1.0 / class_counts[y_train_labels]
        sampler = WeightedRandomSampler(
            torch.tensor(sample_weights, dtype=torch.float),
            num_samples=len(sample_weights),
            replacement=True
        )
        train_loader = DataLoader(train_dataset, batch_size=batch_size, sampler=sampler, num_workers=2, pin_memory=True)
    else:
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)

    # Val and test loaders are typically not shuffled or weighted
    val_loader   = DataLoader(val_dataset,  batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

    print(f"Created loaders - Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")
    print(f"Train Loader batch size: {batch_size}, Weighted Sampling: {config_dict.get('weighted_sampling', False)}")

    return train_loader, val_loader, test_loader

In [17]:

train_transform_final_experiment = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.3, scale=(0.02, 0.1))
])


val_transform_final_experiment = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

experiment_train_loader, experiment_val_loader, _ = create_experiment_loaders(
    X_train, y_train, X_val, y_val, X_test, y_test, # Pass raw data
    train_transform_final_experiment, val_transform_final_experiment, # Pass appropriate transforms
    config_final_match # Pass the experiment config
)



Created loaders - Train: 28709 | Val: 3589 | Test: 3589
Train Loader batch size: 64, Weighted Sampling: True


In [ ]:
config_final_match = {
    'architecture':       'ResNet18',
    'variant':            'RicherClassifier_LabelSmoothing_Weighted',
    'lr':                 1e-4,
    'batch_size':         64,
    'optimizer':          'Adam',
    'epochs':             25,
    'dropout_rate':       0.5,
    'weight_decay':       1e-4,
    'freeze_backbone':    False,
    'early_stopping':     True,
    'early_stop_patience': 4,
    'label_smoothing':    0.1,
    'weighted_sampling':  True
}

model_final_match = FERResNetV2(dropout_rate=0.5, freeze_backbone=False)

acc_final_match = run_experiment_v3(
    model_final_match,
    '18_ResNet18_RC_LB_W',
    config_final_match,
    train_loader_final,
    val_loader_final,
    device
)

wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


Ep 01/25 | Train 51.6% (1.4296) | Val 58.5% (1.3116) | Gap -6.9% | GradNorm 3.501
Ep 02/25 | Train 65.0% (1.1848) | Val 62.5% (1.2367) | Gap 2.5% | GradNorm 3.325
Ep 03/25 | Train 68.6% (1.1171) | Val 63.9% (1.1938) | Gap 4.7% | GradNorm 3.297
Ep 04/25 | Train 71.3% (1.0600) | Val 64.0% (1.2065) | Gap 7.3% | GradNorm 3.234
Ep 05/25 | Train 73.2% (1.0239) | Val 64.5% (1.2035) | Gap 8.7% | GradNorm 3.327
Ep 06/25 | Train 75.0% (0.9914) | Val 64.9% (1.1790) | Gap 10.2% | GradNorm 3.364
Ep 07/25 | Train 76.8% (0.9557) | Val 65.7% (1.1921) | Gap 11.0% | GradNorm 3.450
Ep 08/25 | Train 78.6% (0.9212) | Val 67.5% (1.1788) | Gap 11.1% | GradNorm 3.537
Ep 09/25 | Train 79.4% (0.8979) | Val 67.1% (1.1934) | Gap 12.3% | GradNorm 3.698
Ep 10/25 | Train 80.0% (0.8852) | Val 66.5% (1.1963) | Gap 13.5% | GradNorm 3.767
Ep 11/25 | Train 81.5% (0.8638) | Val 66.7% (1.2121) | Gap 14.8% | GradNorm 3.828
Ep 12/25 | Train 84.6% (0.7989) | Val 68.3% (1.1972) | Gap 16.4% | GradNorm 3.896
Ep 13/25 | Train 86.

acc_gap,▁▃▄▄▅▅▅▅▅▆▆▆▇▇▇▇▇████
best_val_acc,▁
early_stop_counter,▁▁▁▁▁▁▁▁▁▃▆█▁▁▃▆█▁▃▆█
epoch,▁▁▂▂▂▃▃▃▄▄▅▅▅▆▆▆▇▇▇██
epochs_trained,▁
grad_norm,▃▂▁▁▂▂▂▃▄▄▅▅▆▇▇█▇▇▇█▇
lr,██████████▄▄▄▄▄▂▂▂▂▁▁
train_acc,▁▃▄▄▅▅▅▅▆▆▆▆▇▇▇▇█████
train_loss,█▆▅▅▅▄▄▄▃▃▃▃▂▂▂▂▂▁▁▁▁
val_acc,▁▄▄▅▅▅▆▇▆▆▆▇█▇▇▇█████
+1,...



-> Best val acc: 69.57%


In [5]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset
from torchvision import models
from PIL import Image

class FERDatasetFinal(Dataset):
    def __init__(self, imgs, lbls, transform=None):
        self.imgs      = imgs
        self.lbls      = lbls
        self.transform = transform

    def __len__(self):
        return len(self.imgs)

    def __getitem__(self, idx):
        img = Image.fromarray(self.imgs[idx])
        if self.transform:
            img = self.transform(img)
        return img, int(self.lbls[idx])

class FERResNetV2Final(nn.Module):
    def __init__(self, num_classes=7, dropout_rate=0.5, freeze_backbone=False):
        super().__init__()
        self.backbone = models.resnet18(weights='IMAGENET1K_V1')

        if freeze_backbone:
            for param in self.backbone.parameters():
                param.requires_grad = False

        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Sequential(
            nn.Linear(in_features, 256),
            nn.ReLU(),
            nn.Dropout(p=dropout_rate),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        return self.backbone(x)

print("Dataset and Model classes are ready!")

Dataset and Model classes are ready!


In [6]:
import torch.nn.utils as utils
import wandb

class EarlyStoppingFinal:
    def __init__(self, patience=4, min_delta=0.001):
        self.patience     = patience
        self.min_delta    = min_delta
        self.best_acc     = 0.0
        self.counter      = 0
        self.best_weights = None
        self.should_stop  = False

    def step(self, val_acc, model):
        if val_acc > self.best_acc + self.min_delta:
            self.best_acc     = val_acc
            self.counter      = 0
            self.best_weights = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.should_stop = True

    def restore_best(self, model):
        if self.best_weights:
            model.load_state_dict(self.best_weights)

def train_epoch_final(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct, total, total_grad_norm = 0.0, 0, 0, 0.0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs   = model(images)
        loss      = criterion(outputs, labels)
        loss.backward()
        grad_norm = utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        total_grad_norm += grad_norm.item()
        optimizer.step()
        total_loss += loss.item() * images.size(0)
        correct    += (outputs.argmax(1) == labels).sum().item()
        total      += labels.size(0)
    return total_loss / total, correct / total, total_grad_norm / len(loader)

def evaluate_epoch_final(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.inference_mode():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs    = model(images)
            loss       = criterion(outputs, labels)
            total_loss += loss.item() * images.size(0)
            correct    += (outputs.argmax(1) == labels).sum().item()
            total      += labels.size(0)
    return total_loss / total, correct / total

def run_experiment_v4(model, run_name, config, train_loader, val_loader, device):
    wandb.init(project='fer-challenge', name=run_name, group=config['architecture'], config=config, reinit=True)
    model = model.to(device)
    criterion = nn.CrossEntropyLoss(label_smoothing=config.get('label_smoothing', 0.0))
    optimizer = torch.optim.Adam(model.parameters(), lr=config['lr'], weight_decay=config.get('weight_decay', 0.0))

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)

    stopper = EarlyStoppingFinal(patience=config.get('early_stop_patience', 4)) if config.get('early_stopping', False) else None
    best_val_acc = 0.0

    for epoch in range(1, config['epochs'] + 1):
        train_loss, train_acc, grad_norm = train_epoch_final(model, train_loader, criterion, optimizer, device)
        val_loss,   val_acc              = evaluate_epoch_final(model, val_loader, criterion, device)

        scheduler.step(val_acc)
        current_lr = optimizer.param_groups[0]['lr']
        gap        = train_acc - val_acc

        wandb.log({
            'epoch': epoch, 'train_loss': train_loss, 'val_loss': val_loss,
            'train_acc': train_acc, 'val_acc': val_acc, 'acc_gap': gap, 'lr': current_lr
        })

        print(f'Ep {epoch:02d} | Train {train_acc*100:.1f}% | Val {val_acc*100:.1f}% | Gap {gap*100:.1f}%')

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), f'/content/{run_name}_best.pt')

        if stopper is not None:
            stopper.step(val_acc, model)
            if stopper.should_stop:
                print(f'Early stopping at epoch {epoch}!')
                stopper.restore_best(model)
                break

    wandb.finish()
    print(f'\n-> Best val acc: {best_val_acc*100:.2f}%')
    return best_val_acc

print("Training engine (v4) is ready!")

Training engine (v4) is ready!


In [7]:
import pandas as pd
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import transforms

train_transform_final = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.3, scale=(0.02, 0.1))
])

val_transform_final = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset_clean = FERDatasetFinal(X_train, y_train, transform=train_transform_final)
val_dataset_clean   = FERDatasetFinal(X_val,   y_val,   transform=val_transform_final)

class_counts   = pd.Series(y_train).value_counts().sort_index().values
sample_weights = 1.0 / class_counts[y_train]
sampler_clean  = WeightedRandomSampler(
    torch.tensor(sample_weights, dtype=torch.float),
    num_samples=len(sample_weights),
    replacement=True
)

train_loader_final = DataLoader(train_dataset_clean, batch_size=64, sampler=sampler_clean, num_workers=2, pin_memory=True)
val_loader_final   = DataLoader(val_dataset_clean,   batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

print("Clean loaders ready!")

Clean loaders ready!


In [8]:
config_final = {
    'architecture':        'ResNet18',
    'variant':             'Final_Clean_Experiment',
    'lr':                  1e-4,
    'batch_size':          64,
    'optimizer':           'Adam',
    'epochs':              25,
    'dropout_rate':        0.5,
    'weight_decay':        1e-4,
    'freeze_backbone':     False,
    'early_stopping':      True,
    'early_stop_patience': 4,
    'label_smoothing':     0.1,
    'weighted_sampling':   True
}

model_final = FERResNetV2Final(dropout_rate=0.5, freeze_backbone=False)

print("Launching the final experiment...")
acc_final = run_experiment_v4(
    model_final,
    '20_ResNet18_FINAL_CLEAN',
    config_final,
    train_loader_final,
    val_loader_final,
    device
)

Launching the final experiment...


wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


Ep 01 | Train 50.6% | Val 58.3% | Gap -7.7%
Ep 02 | Train 64.4% | Val 62.8% | Gap 1.6%
Ep 03 | Train 68.9% | Val 64.3% | Gap 4.5%
Ep 04 | Train 71.7% | Val 65.0% | Gap 6.7%
Ep 05 | Train 73.7% | Val 66.8% | Gap 6.9%
Ep 06 | Train 75.2% | Val 65.6% | Gap 9.6%
Ep 07 | Train 76.9% | Val 66.7% | Gap 10.2%
Ep 08 | Train 78.3% | Val 66.1% | Gap 12.3%
Ep 09 | Train 81.4% | Val 68.9% | Gap 12.5%
Ep 10 | Train 83.5% | Val 68.3% | Gap 15.2%
Ep 11 | Train 84.8% | Val 68.7% | Gap 16.2%
Ep 12 | Train 85.9% | Val 68.9% | Gap 17.0%
Ep 13 | Train 87.8% | Val 69.5% | Gap 18.3%
Ep 14 | Train 88.9% | Val 69.3% | Gap 19.6%
Ep 15 | Train 89.5% | Val 69.2% | Gap 20.3%
Ep 16 | Train 90.1% | Val 68.6% | Gap 21.5%
Ep 17 | Train 91.2% | Val 69.2% | Gap 21.9%
Early stopping at epoch 17!


acc_gap,▁▃▄▄▄▅▅▆▆▆▇▇▇▇███
epoch,▁▁▂▂▃▃▄▄▅▅▅▆▆▇▇██
lr,███████▄▄▄▄▂▂▂▂▁▁
train_acc,▁▃▄▅▅▅▆▆▆▇▇▇▇████
train_loss,█▆▅▄▄▄▄▃▃▂▂▂▂▁▁▁▁
val_acc,▁▄▅▅▆▆▆▆█▇▇████▇█
val_loss,█▅▃▂▁▂▂▃▁▂▁▃▂▂▂▃▃
acc_gap,0.21945
epoch,17
lr,1e-05
train_acc,0.91156



-> Best val acc: 69.49%
